In [1]:
import sys 
from pathlib import Path 
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd 

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
admissions_df = pd.read_csv(RAW_DATA_DIR / "admissions.csv")
icustays_df = pd.read_csv(RAW_DATA_DIR / "icustays.csv")
patients_df = pd.read_csv(RAW_DATA_DIR / "patients.csv")

In [3]:
icustays_df_drop_cols = [
    "last_careunit"
]

icustays_df = icustays_df.drop(
    columns=icustays_df_drop_cols
)

In [4]:
admissions_df_drop_cols = [
    "dischtime",
    "deathtime",
    "edouttime",
    "edregtime",
    "discharge_location",
    "hospital_expire_flag",
    "admit_provider_id"
]

admissions_df = admissions_df.drop(
    columns=admissions_df_drop_cols
)

In [5]:
patients_df_drop_cols = [
    "anchor_year",
    "anchor_year_group",
    "dod"
]

patients_df = patients_df.drop(
    columns=patients_df_drop_cols
)

In [6]:
icustays_df = icustays_df.merge(
    patients_df,
    how="inner",
    on="subject_id"
)

icustays_df = icustays_df.merge(
    admissions_df,
    how="inner",
    on=["subject_id", "hadm_id"]
)

print(icustays_df.shape)
print(icustays_df.isna().sum())
print(icustays_df.duplicated().sum())

(94458, 16)
subject_id               0
hadm_id                  0
stay_id                  0
first_careunit           0
intime                   0
outtime                 14
los                     14
gender                   0
anchor_age               0
admittime                0
admission_type           0
admission_location       0
insurance             1523
language               396
marital_status        7761
race                     0
dtype: int64
0


In [7]:
icustays_df = icustays_df.dropna(
    subset=["outtime", "los"]
)

print(icustays_df.isna().sum())

subject_id               0
hadm_id                  0
stay_id                  0
first_careunit           0
intime                   0
outtime                  0
los                      0
gender                   0
anchor_age               0
admittime                0
admission_type           0
admission_location       0
insurance             1523
language               396
marital_status        7756
race                     0
dtype: int64


In [8]:
categorical_cols = ["insurance", "language", "marital_status"]
icustays_df[categorical_cols] = icustays_df[categorical_cols].fillna("Unknown")

print(icustays_df.isna().sum())
print(icustays_df.duplicated(subset=["subject_id", "hadm_id", "stay_id"]).sum())

subject_id            0
hadm_id               0
stay_id               0
first_careunit        0
intime                0
outtime               0
los                   0
gender                0
anchor_age            0
admittime             0
admission_type        0
admission_location    0
insurance             0
language              0
marital_status        0
race                  0
dtype: int64
0


In [9]:
print(icustays_df.shape)
icustays_df.to_parquet(PROCESSED_DATA_DIR / "icustays_clean.parquet", index=False)

(94444, 16)
